# 03 · Stage 2a — benign subtype {nv, bkl, vasc} (multimodal)

Given a benign decision at Stage 1, classify the benign subtype. `df` (dermatofibroma, ~115 imgs) is grouped benign at Stage 1 but not sub-classified here — a documented limitation.

**Design.** We train **two** models on the *same* EfficientNetB3 backbone so any
performance gap is attributable to the metadata, not a different vision model:

1. **Image-only baseline** — the original project's architecture (the control).
2. **Multimodal late-fusion** — image embedding ⊕ MLP over age/sex/site.

Both are trained on the **full** stage data with **balanced class weights** and
on-the-fly augmentation — no undersampling (the key methodological fix). Models
are saved to `models/` with names the inference `pipeline.py` expects.

---
*Part of: **A Multimodal, Explainable Hierarchical Framework for Skin-Lesion
Classification with Diffusion-Based Augmentation of Rare Malignancies.***
All training/eval logic lives in the tested modules under `src/`; these
notebooks orchestrate them on a Colab T4.

In [1]:
!pip -q install scikit-learn matplotlib kaggle >/dev/null 2>&1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'
#
LOCAL_PROJECT = '/content/Skin_Cancer_Research_Grant2024'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

folders = ['data']

for folder in folders:
    source = os.path.join(DRIVE_PROJECT, folder)
    destination = os.path.join(LOCAL_PROJECT, folder)

    print(f"Copying {folder}...")

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

    print(f"✓ {folder} copied")

print("\nAll folders copied to Colab!")

In [17]:
import os, sys

PROJECT_DIR = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'

SRC       = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR= os.path.join(PROJECT_DIR, 'models')
DATA_DIR  = '/content/Skin_Cancer_Research_Grant2024/data'
FIG_DIR   = os.path.join(PROJECT_DIR, 'report', 'figures')
SPLIT_CSV = os.path.join(MODELS_DIR, 'ham_split.csv')
PREP_JSON = os.path.join(MODELS_DIR, 'preprocessing.json')
for d in (MODELS_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isdir(SRC), f"src/ not found at {SRC} — set PROJECT_DIR correctly"
sys.path.insert(0, SRC)
print("PROJECT_DIR =", PROJECT_DIR)

PROJECT_DIR = /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024


In [18]:
import os, json
import pandas as pd
import data as D, metadata as M, model as Model

STAGE = "benign"
EPOCHS = 30
BATCH = 32

proc = M.MetadataProcessor.load(PREP_JSON)
df = pd.read_csv(SPLIT_CSV)
sdf = D.filter_for_stage(df, STAGE)
tr = sdf[sdf.split == 'train']; va = sdf[sdf.split == 'val']; te = sdf[sdf.split == 'test']
n_classes = len(D.STAGE_CLASSES[STAGE])
class_weight = D.compute_class_weights(tr, STAGE)
print("classes:", D.STAGE_CLASSES[STAGE], "| n_classes:", n_classes)
print("train/val/test:", len(tr), len(va), len(te))
print("class weights:", class_weight)

classes: ['nv', 'bkl', 'vasc'] | n_classes: 3
train/val/test: 5740 1013 1193
class weights: {0: 0.394990366088632, 1: 2.4097397145256085, 2: 18.758169934640524}


In [20]:
# ---------------- Image-only baseline (control) ----------------
tf_tr = D.make_dataset(tr, STAGE, None, multimodal=False, batch_size=BATCH, training=True, augment=True)
tf_va = D.make_dataset(va, STAGE, None, multimodal=False, batch_size=BATCH, training=False)

baseline = Model.compile_model(Model.build_image_only_baseline(n_classes))
base_path = os.path.join(MODELS_DIR, "benign_baseline.keras")
hist_b = baseline.fit(tf_tr, validation_data=tf_va, epochs=EPOCHS,
                      class_weight=class_weight,
                      callbacks=Model.make_callbacks(base_path))
baseline.save(base_path)
json.dump(hist_b.history, open(base_path.replace('.keras', '_history.json'), 'w'))
print("saved baseline ->", base_path)

Epoch 1/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 429s 2s/step - accuracy: 0.4601 - loss: 1.2357 - val_accuracy: 0.7374 - val_loss: 0.7217 - learning_rate: 1.0000e-04
Epoch 2/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 158s 880ms/step - accuracy: 0.5512 - loss: 0.8260 - val_accuracy: 0.7266 - val_loss: 0.6794 - learning_rate: 1.0000e-04
Epoch 3/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 159s 882ms/step - accuracy: 0.6037 - loss: 0.7026 - val_accuracy: 0.7098 - val_loss: 0.7048 - learning_rate: 1.0000e-04
Epoch 4/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 175s 973ms/step - accuracy: 0.6347 - loss: 0.6513 - val_accuracy: 0.7335 - val_loss: 0.6903 - learning_rate: 1.0000e-04
Epoch 5/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 156s 869ms/step - accuracy: 0.6672 - loss: 0.5802 - val_accuracy: 0.7532 - val_loss: 0.6620 - learning_rate: 1.0000e-04
Epoch 6/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 160s 889ms/step - accuracy: 0.6939 - loss: 0.5272 - val_accuracy: 0.7720 - val_loss: 0.6251 - learning_rate: 5.4881e-05
Epoch 7/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 155

In [21]:
# ---------------- Multimodal late-fusion ----------------
tf_tr_m = D.make_dataset(tr, STAGE, proc, multimodal=True, batch_size=BATCH, training=True, augment=True)
tf_va_m = D.make_dataset(va, STAGE, proc, multimodal=True, batch_size=BATCH, training=False)

mm = Model.compile_model(Model.build_multimodal_model(n_classes, proc.feature_dim))
mm_path = os.path.join(MODELS_DIR, "benign_multimodal.keras")
hist_m = mm.fit(tf_tr_m, validation_data=tf_va_m, epochs=EPOCHS,
                class_weight=class_weight,
                callbacks=Model.make_callbacks(mm_path))
mm.save(mm_path)
json.dump(hist_m.history, open(mm_path.replace('.keras', '_history.json'), 'w'))
print("saved multimodal ->", mm_path)

Epoch 1/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 362s 1s/step - accuracy: 0.3730 - loss: 1.3087 - val_accuracy: 0.7364 - val_loss: 0.6636 - learning_rate: 1.0000e-04
Epoch 2/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 161s 891ms/step - accuracy: 0.4815 - loss: 0.9956 - val_accuracy: 0.7414 - val_loss: 0.6583 - learning_rate: 1.0000e-04
Epoch 3/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 170s 943ms/step - accuracy: 0.5413 - loss: 0.8875 - val_accuracy: 0.7542 - val_loss: 0.6127 - learning_rate: 1.0000e-04
Epoch 4/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 173s 957ms/step - accuracy: 0.5763 - loss: 0.8047 - val_accuracy: 0.7808 - val_loss: 0.5662 - learning_rate: 1.0000e-04
Epoch 5/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 193s 908ms/step - accuracy: 0.6298 - loss: 0.7001 - val_accuracy: 0.7878 - val_loss: 0.5343 - learning_rate: 1.0000e-04
Epoch 6/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 169s 939ms/step - accuracy: 0.6540 - loss: 0.6408 - val_accuracy: 0.7947 - val_loss: 0.5218 - learning_rate: 5.4881e-05
Epoch 7/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 170

In [22]:
# Quick val-accuracy comparison (full metrics come in notebook 06).
print("baseline   best val_acc: %.4f" % max(hist_b.history['val_accuracy']))
print("multimodal best val_acc: %.4f" % max(hist_m.history['val_accuracy']))

baseline   best val_acc: 0.7828
multimodal best val_acc: 0.8026
